# **Elastic Pendulum**
## FEM Implementation using NGSolve
---------------------------------------

#### **Problem Description**

- Elastic pendulum with large deformations and wall contact using Neo-Hookean material model.
- Currently pendulum and wall using the same material properties.
- Pendulum is fixed at the top and swings under the influence of gravity, initial velocity, and initial angular acceleration (if defined) around the pivot joint (z-axis).
- The wall is fixed at the top and bottom edges.
- A contact condition is defined between the pendulum head edge and the $X+$ edge of the wall.


|Pendulum Geometry|Boundary and Initial Condition|
|-----------------|------------------------------|
|![](images/img_dimensions.png)|![](images/img_setup.png)|

In [1]:
import imp
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from ngsolve.solvers import Newton

import ipywidgets as widgets

C:\Users\frech\AppData\Local\Temp\ipykernel_15672\3504751038.py:1: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp


### Configuration

In [2]:
class GeometryParameters:
    # Pendulum geometry
    r_rod:    float = 0.05
    r_hole:   float = 0.1
    r_head:   float = 0.2
    l_center: float = 0.8
    
    # Wall geometry
    q_wall_deg: float = 0
    wall_len_x: float = 0.15
    wall_len_y: float = 0.8
    wall_len_z: float = 0.1
    

class MaterialParameters:
    material_law:      str   = "linear_elastic" # "linear_elastic" or "neo_hookean"
    E_pendulum:        float = 2.10e9     
    nu_pendulum:       float = 0.35       
    rho_pendulum:      float = 1040      
    E_wall:            float = 2.10e9     
    nu_wall:           float = 0.35       
    rho_wall:          float = 1040      
    thickness:         float = 0.1       
    contact_stiffness: float = 1e9       
    
class MeshParameters:
    max_element_size:  float = 0.035
    mesh_order:        int   = 3
    curved_elements:   bool  = True
    refinement_levels: int   = 0
    

class InitialConditionParameters:
    angular_position_deg:  float = 5
    angular_velocity:      float = -0.1
    angular_acceleration:  float = -0.01
    

class SimulationParameters:
    t_start:  float = 0 
    tau:      float = 0.025  
    t_end:    float = 1       
    
class AnimationParameters:
    interval: int   = 10       
    speed:    float = 3.0      

### Pendulum Class

In [3]:
class SimpleFEMPendulum:
    def __init__(self):
        # Parameters
        self.geom_params = GeometryParameters()
        self.mat_params  = MaterialParameters()
        self.mesh_params = MeshParameters()
        self.init_params = InitialConditionParameters()
        self.sim_params  = SimulationParameters()
        self.anim_params = AnimationParameters()
        
        # Internal states
        self._mesh = None
        self._fes = None
        self._contact = None
        self._material_law = None
        self._simulation_results = []
        
        # Grid Functions
        self._gf_u = None
        self._gf_v = None
        self._gf_a = None
        self._gf_uold = None
        self._gf_vold = None
        
        # Results history
        self._gf_u_history = None
        self._gf_v_history = None
        
        self._setup_material_law()

        pass
    
    def _setup_material_law(self):
        self.E_p, self.E_w     = self.mat_params.E_pendulum, self.mat_params.E_wall
        self.nu_p, self.nu_w   = self.mat_params.nu_pendulum, self.mat_params.nu_wall
        self.rho_p, self.rho_w = self.mat_params.rho_pendulum, self.mat_params.rho_wall
        
        # Lamé parameters
        self.mu_p = self.E_p / 2 / (1+self.nu_p)
        self.lam_p = self.E_p * self.nu_p / ((1+self.nu_p)*(1-2*self.nu_p))
        self.mu_w = self.E_w / 2 / (1+self.nu_w)
        self.lam_w = self.E_w * self.nu_w / ((1+self.nu_w)*(1-2*self.nu_w))
        
        if self.mat_params.material_law == "linear_elastic":
            self._deformation_tensor = self.eps
            self._material_law = self.linear_elastic
        
        elif self.mat_params.material_law == "neo_hookean":
            self._deformation_tensor = self.C
            self._material_law = self.neo_hooke
        pass
    
    def create_mesh(self):
        self._create_geometry()
        
        mp = self.mesh_params

        self._mesh = Mesh(OCCGeometry(self._geo, dim=2).GenerateMesh(maxh=mp.max_element_size))

        if mp.curved_elements:
            self._mesh.Curve(mp.mesh_order)
        
        for _ in range(mp.refinement_levels):
            self._mesh.Refine()
    
    def _create_geometry(self):
        gp = self.geom_params

        bar = MoveTo(-gp.r_rod,0).Rectangle(2*gp.r_rod, gp.l_center).Face()
        bar.edges.Min(Y).name="rotation"
        bar.faces.name="bar"
        bar.faces.maxh=gp.r_rod/2

        hole = Circle((0, gp.l_center), gp.r_hole).Face()
        hole.faces.name="hole"

        circ = Circle((0, gp.l_center), gp.r_head).Face()
        circ.edges.maxh=gp.r_head/20
        circ.faces.name="circ"
        circ.faces.maxh=gp.r_head/5
        circ.edges.name="contact_head"
        
        head = circ - hole
        pendulum = head + bar - hole

        pendulum = pendulum.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), 180)
        pendulum.name = "pendulum"

        # Wall
        wall_pos_x = -gp.r_head - gp.wall_len_x
        wall_pos_y = -gp.l_center - gp.wall_len_y/2
        wall = MoveTo(wall_pos_x, wall_pos_y).Rectangle(gp.wall_len_x, gp.wall_len_y).Face()
        wall.faces.maxh = gp.r_head/5
        wall.edges.Max(X).name = "contact_wall"
        wall.edges.Max(X).maxh = gp.r_head/20
        wall.edges.Max(Y).name = "fix"
        wall.edges.Min(Y).name = "fix"
        wall.name = "wall"

        self._geo = Compound([pendulum, wall])
    
    def initialize(self):
        self._initialize_fe_spaces()
        self._initialize_grid_functions()
        self._set_initial_conditions()
        self._initialize_contact()
        self._setup_bilinear_form()
        pass
    
    def _initialize_fe_spaces(self):
        # Create H1 vector space for 3D quantities (displacement, velocity, acceleration)
        self._V = VectorH1(self._mesh, order=self.mesh_params.mesh_order, dirichlet="fix")
        
        # Create NumberSpace for Lagrange multipliers (rotation constraint)
        self._Q = NumberSpace(self._mesh, definedon=self._mesh.Boundaries('rotation'))
        
        # Mixed FE space
        self._fes = self._V * self._Q**2
        (self._u, self._q), (self._v, self._p) = self._fes.TnT()
        
        # Scalar H1 space for stress
        self._S = H1(self._mesh, order=3)
        
        self._setup_material_law()      
        
        pass
    
    def _initialize_grid_functions(self):
        # Initialize grid functions
        self._gf_u = GridFunction(self._fes)  # Current state
        self._gf_v = GridFunction(self._fes)  # Velocity
        self._gf_a = GridFunction(self._fes)  # Acceleration
        
        self._gf_uold = GridFunction(self._fes)  # Previous displacement
        self._gf_vold = GridFunction(self._fes)  # Previous velocity
        self._gf_aold = GridFunction(self._fes)  # Previous acceleration
        
        self._gf_vm_p = GridFunction(self._S) # Von Mises - pendulum
        self._gf_vm_w = GridFunction(self._S) # Von Mises - wall              
        
        # Time series storage
        self._gf_u_history = GridFunction(self._V, multidim=0)
        self._gf_v_history = GridFunction(self._V, multidim=0)
        self._gf_stress_history = GridFunction(H1(self._mesh, order=3), multidim=0)
    
    def _set_initial_conditions(self):
        icp = self.init_params
        theta = np.deg2rad(icp.angular_position_deg)   # initial angular position
        omega = icp.angular_velocity                   # initial angular velocity
        alpha = icp.angular_acceleration               # initial angular acceleration
        
        c, s = np.cos(theta), np.sin(theta)
        
        # Rotation center (could be made configurable)
        cx = cy = 0.0
        
        # reference coordinates relative to rotation center
        X_rel = CF((x - cx, y - cy))
        
        # rotated radius r0 = R(theta) * (X - P)
        r0 = CF((c * X_rel[0] - s * X_rel[1],
                 s * X_rel[0] + c * X_rel[1]))
        
        # displacement for initial position
        u0 = CF((r0[0] - X_rel[0],
                 r0[1] - X_rel[1]))
        
        # Initial velocity: v0 = omega x r0
        v0 = CF((-omega * r0[1],
                  omega * r0[0]))
        
        # Initial acceleration: a0 = alpha x r0 - omega^2 * r0
        a0 = CF(( -alpha * r0[1],
               alpha * r0[0] ))
        
        # Set initial conditions
        self._gf_u.components[0].Set(u0, definedon=self._mesh.Materials("pendulum"))
        self._gf_v.components[0].Set(v0, definedon=self._mesh.Materials("pendulum"))
        self._gf_a.components[0].Set(a0, definedon=self._mesh.Materials("pendulum"))

        # Copy to "old" variables
        self._gf_uold.vec[:] = self._gf_u.vec
        self._gf_vold.vec[:] = self._gf_v.vec
        self._gf_aold.vec[:] = self._gf_a.vec
        
        # Add to history
        self._gf_u_history.AddMultiDimComponent(self._gf_u.components[0].vec)
        self._gf_v_history.AddMultiDimComponent(self._gf_v.components[0].vec)
        pass
    
    def _initialize_contact(self):
        k_n = self.mat_params.contact_stiffness
            
        # Define slave / master contact set
        master = self._mesh.Boundaries("contact_wall") # fixed wall
        slave = self._mesh.Boundaries("contact_head") # mobing head
        self._contact = ContactBoundary(slave, master)
        
        # Geometry and normal vector
        X = CoefficientFunction((x,y))         # reference coords on master
        X_M = X.Other()                        # reference coords on slave
        nM = specialcf.normal(2).Other()                # outer normal on master
        
        u = self._u                                  # trial displacement on master
        u_M = self._u.Other()                         # trial displacement on slave
        u_old = self._gf_uold.components[0]             # previous displacement on master
        u_M_old = self._gf_uold.components[0].Other()    # previous displacement on slave
        
        # Absolute gap (current configuration)
        delta_cur = (X + u) - (X_M + u_M)
        self._gap_abs_cf = InnerProduct(delta_cur, nM)
        
        # Incremental gap over the current time step
        delta_old = (X + u_old) - (X_M + u_M_old)
        g_abs_old = InnerProduct(delta_old, nM)
        self._gap_inc_cf = self._gap_abs_cf - g_abs_old
        
        # Contact Penalty Energy
        self._contact.AddEnergy(
            IfPos(-self._gap_abs_cf,
                  k_n * self._gap_abs_cf * self._gap_abs_cf, 0),
            deformed=True)
        
        # Monitoring spaces to sample gap values on the slave boundary
        self._SL2_gap = SurfaceL2(self._mesh, order=1, definedon=slave) # boundary scalar space
        self._gf_gap_abs = GridFunction(self._SL2_gap)
        self._gf_gap_inc = GridFunction(self._SL2_gap)
        
        # Time series
        self._gap_abs_min_hist = []
        self._gap_inc_min_hist = []
        
        #gap_function = (X + self._u-self._uold - (X.Other() + self._u.Other() - self._uold.Other())) * (-specialcf.normal(2).Other())   
        #self._contact.AddEnergy(IfPos(gap_function, k_n*gap_function*gap_function, 0), deformed=True)
        
    
    def _setup_bilinear_form(self):
        # Bilinear form
        self._bfa = BilinearForm(self._fes)
        
        self._bfa += Variation(self._material_law(self._deformation_tensor(self._u), 
                                                  self.mu_p, self.lam_p)*dx("pendulum")).Compile()
        
        self._bfa += Variation(self._material_law(self._deformation_tensor(self._u),
                                            self.mu_w, self.lam_w)*dx("wall")).Compile()
        
        # Rotation constraint
        self._bfa += (InnerProduct(self._u, self._p) + InnerProduct(self._v, self._q)) * ds('rotation')
        
        self.tau = self.sim_params.tau
        vel_new = 2/self.tau * (self._u-self._gf_uold.components[0]) - self._gf_vold.components[0]
        acc_new = 2/self.tau * (vel_new-self._gf_vold.components[0]) - self._gf_aold.components[0]
        
        rhoA_p = self.rho_p * self.mat_params.thickness
        rhoA_w = self.rho_w * self.mat_params.thickness
        g = 9.81
        
        # inertia (mass matrix effect)
        self._bfa += rhoA_p * InnerProduct(acc_new, self._v) * dx("pendulum")
        self._bfa += rhoA_w * InnerProduct(acc_new, self._v) * dx("wall")
        
        # gravity force
        self._bfa += InnerProduct(CF((0, rhoA_p*g)), self._v) * dx("pendulum")
        self._bfa += InnerProduct(CF((0, rhoA_w*g)), self._v) * dx("wall")
        
        #force = CF((0, -1))
        #force = CF((0, -g * self.mat_params.rho_pendulum))

        # need to add to the bilinear form since it depends on the current valurs of the GridFunctions
        #self._bfa += rhoA * InnerProduct(acc_new, self._v)*dx
        #self._bfa += -force*self._v*dx
        # self._bfa += acc_new * self._v*dx
        # self._bfa += -force*self._v*dx
        
    def simulate(self):
        t = self.sim_params.t_start
        self.tend = self.sim_params.t_end
        i = 1
        with TaskManager():
            while t < self.tend:
                i += 1
                t += self.tau

                # A) Update contact with the current displacement
                self._contact.Update(self._gf_u.components[0], self._bfa)
                #self._contact.Update(self._uold, self._bfa)
                
                # B) Solve nonlinear system with Newton               
                Newton(a=self._bfa, u=self._gf_u, printing=False, inverse="sparsecholesky")

                # C) Update kinematic variables (velocity, acceleration)
                self._gf_v.vec[:] = 2/self.tau * (self._gf_u.vec-self._gf_uold.vec) - self._gf_vold.vec
                self._gf_a.vec[:] = 2/self.tau * (self._gf_v.vec-self._gf_vold.vec) - self._gf_aold.vec
                
                # # D) Sample gaps on the slave boundary
                # self._gf_gap_abs.Set(self._gap_abs_cf)
                # vals_abs = self._gf_gap_abs.vec.FV().NumPy()
                # gap_abs_min = float(vals_abs.min()) if vals_abs.size else float('nan')
                
                # # E) Sample incremental gap on the slave boundary
                # self._gf_gap_inc.Set(self._gap_inc_cf)
                # vals_inc = self._gf_gap_inc.vec.FV().NumPy()
                # gap_inc_min = float(vals_inc.min()) if vals_inc.size else float('nan')

                # self._gap_abs_min_hist.append(gap_abs_min)
                # self._gap_inc_min_hist.append(gap_inc_min)

                # F) Update "old" variables for next time step
                self._gf_uold.vec[:] = self._gf_u.vec
                self._gf_vold.vec[:] = self._gf_v.vec
                self._gf_aold.vec[:] = self._gf_a.vec
                
                # G) Store results in time series
                self._gf_u_history.AddMultiDimComponent(self._gf_u.components[0].vec)
                self._gf_v_history.AddMultiDimComponent(self._gf_v.components[0].vec)
                
                # H) Compute stress, and print info every n-th step
                if i % self.anim_params.interval == 0:
                    print(f't = {t:6.3f}')
                    # Penetration info
                    #pen = max(0.0, -gap_abs_min)
                                        
                    # Compute stress from current displacement
                    # C_ = self.C(self._gf_u.components[0]).MakeVariable()
                    # sigma_p = self.neo_hooke(C_, self.mu_p, self.lam_p).Diff(C_)
                    # sigma_w = self.neo_hooke(C_, self.mu_w, self.lam_w).Diff(C_)
                    
                    # vm_p = self.von_mises_2d(sigma_p)
                    # vm_w = self.von_mises_2d(sigma_w)
                    
                    # # Set stress for both pendulum and wall
                    # self._gf_vm_p.Set(vm_p, definedon=self._mesh.Materials("pendulum"))
                    # self._gf_vm_w.Set(vm_w, definedon=self._mesh.Materials("wall"))
                    
                    # # Add to history
                    # self._gf_stress_history.AddMultiDimComponent(self._gf_vm_p.vec + self._gf_vm_w.vec)
                
    def visualize(self, mesh=True, u=True, v=True, a=True):
        if mesh:
            # Mesh Geometry
            tw_geometry = widgets.Text(value="Mesh Geometry")
            display(tw_geometry)
            Draw(self._mesh, "mesh")
        if u:
            # Displacement
            tw_displacement = widgets.Text(value="Angular Displacement", fontsize=16, fontweight='bold')
            display(tw_displacement)
            Draw(self._gf_u.components[0], deformation=True)
        if v:
            # Velocity
            tw_velocity = widgets.Text(value="Angular Velocity")
            display(tw_velocity)
            Draw(self._gf_v.components[0], deformation=self._gf_u.components[0], vectors=True)
        if a:
            # Acceleration
            tw_acceleration = widgets.Text(value="Tangential Angular Acceleration")
            display(tw_acceleration)
            Draw(self._gf_a.components[0], deformation=self._gf_u.components[0], vectors=True)
        
    def animate_u(self):
        settings = {"Multidim": {
                    "speed" : self.anim_params.speed
                }};
        
        tw_u = widgets.Text(value="Displacement Animation")
        display(tw_u)

        Draw(self._gf_u_history,
             self._mesh,
             interpolate_multidim=True,
             deformation=self._gf_u_history,
             animate=True,
             autoscale = False,
             min = 0, max = 1,
             settings = settings);
        
    def animate_stress(self):
        settings = {"Multidim": {
                    "speed" : self.anim_params.speed
                }};
        
        tw_stress = widgets.Text(value="Stress History Animation")
        display(tw_stress)
        
        # Animate stress history and deformation history in one scene
        Draw(self._gf_stress_history,
             self._mesh,
             interpolate_multidim=True,
             deformation=self._gf_u_history,
             animate=True,
             settings = settings);
     
    # Hyperelastic material model: Neo-Hookean   
    def C(self, u):
        F = Id(u.dim) + Grad(u)
        return F.trans * F
        
    def neo_hooke (self, C, mu, lam):
        return 0.5*mu*(Trace(C-Id(self._u.dim)) + 2*mu/lam*Det(C)**(-lam/2/mu)-1)
    
    def sigma_neo_hooke(self, u, lam, mu):
        d = u.dim
        F = Id(d) + Grad(u)
        J = Det(F)
        C = F.trans * F
        
        # Psi = mu/2*(I1 -d) - mu*ln(J) + lam/2*ln(J)^2
        I1 = Trace(C)
        Psi = mu/2 * (I1 - d) - mu * log(J) + lam/2 *log(J)**2
        
        # Second Piola-Kirchhoff stress
        C_var = C.MakeVariable()
        Psi_var = mu/2 * (Trace(C_var) - d) - mu * log(Det(F)) + lam/2 * log(Det(F))**2
        S = Psi_var.Diff(C_var)
        
        # Cauchy stress
        sigma = 1.0 / J * F * S * F.trans
        return sigma
    
    def von_mises_2d(self, sig):
        sxx, syy, sxy = sig[0,0], sig[1,1], sig[0,1]
        return sqrt(sxx*sxx - sxx*syy + syy*syy + 3.0*sxy*sxy)

    # Linear elastic material model
    def eps(self, u):
        return Sym(Grad(u))
    
    def linear_elastic(self, eps, mu, lam):
        return 0.5*lam*Trace(eps)**2 + mu*InnerProduct(eps, eps)
    
    def sigma_linear(self, eps, u, lam, mu):
        return lam*Trace(eps)*Id(u.dim) + 2*mu*eps

### Simulation

In [4]:
myPendulum = SimpleFEMPendulum()
myPendulum.create_mesh()
myPendulum.initialize()
myPendulum.visualize()

Text(value='Mesh Geometry')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Angular Displacement')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Angular Velocity')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Tangential Angular Acceleration')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [5]:
myPendulum.simulate()

t =  0.225
t =  0.475
t =  0.725
t =  0.975


In [6]:
myPendulum.animate_u()

Text(value='Displacement Animation')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Multidim': {'speed': 3.0}},…

In [8]:
myPendulum.animate_stress()

Text(value='Stress History Animation')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Multidim': {'speed': 3.0}},…